<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_Logistic_Regression/blob/main/Internship_LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

In [3]:
df = pd.read_csv("properties_features.csv", parse_dates=["listed_date"])

In [4]:
print(f"loaded {df.shape[0]} rows")

loaded 3000 rows


In [8]:
settled = df[df["sold_fast"].notna()].copy()
settled["sold_fast"] = settled["sold_fast"].astype(int)

print(f"\nall listings        : {len(df)}")
print(f"still Active (dropped): {len(df) - len(settled)}")
print(f"modelling population  : {len(settled)}")
print(f"\nclass balance:")
print(settled["sold_fast"].value_counts().rename({0: "slow (0)", 1:"fast (1)"}))
print(f"positive rate: {settled['sold_fast'].mean():.1%}")


all listings        : 3000
still Active (dropped): 437
modelling population  : 2563

class balance:
sold_fast
slow (0)    1572
fast (1)     991
Name: count, dtype: int64
positive rate: 38.7%


In [9]:
from sklearn.model_selection import train_test_split

FORBIDDEN = ["days_on_market", "sold_fast", "listing_status","price_reduced_count"]
DROP = FORBIDDEN + ["listed_date", "description", "latitude","longitude"]

X = settled.drop(columns=[c for c in DROP if c in settled.columns])
y = settled["sold_fast"]

print(f"\nfeatures available: {X.shape[1]}")
print("\nNOW LEGAL, because the price is already set:")
for c in ["price_aed", "log_price_aed", "price_per_sqft","rental_yield_pct", "area_price_encoded"]:
  print(f"  {c}")

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2, random_state=42, stratify=y)

print(f"\ntrain {len(X_train)} ({y_train.mean():.1%} positive)")
print(f"test  {len(X_test)} ({y_test.mean():.1%} positive)")


features available: 64

NOW LEGAL, because the price is already set:
  price_aed
  log_price_aed
  price_per_sqft
  rental_yield_pct
  area_price_encoded

train 2050 (38.7% positive)
test  513 (38.6% positive)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

def make_preprocessor():
  return ColumnTransformer([
      ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                        ("s", StandardScaler())]), numeric_cols),
      ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("e", OneHotEncoder(handle_unknown="ignore",
                                            sparse_output=False,
                                            min_frequency=10))]), categorical_cols),
    ])
plain = Pipeline([("p", make_preprocessor()),
                  ("m", LogisticRegression(max_iter=3000))]).fit(X_train, y_train)

balanced = Pipeline([ ("p", make_preprocessor()),
                      ("m", LogisticRegression(max_iter=3000,class_weight="balanced"))]).fit(X_train, y_train)

print(f"\nfitted on {len(X_train)} rows x "
      f"{plain.named_steps['p'].transform(X_train).shape[1]} features")


fitted on 2050 rows x 177 features


In [11]:
proba = balanced.predict_proba(X_test)[:, 1]
labels = balanced.predict(X_test)

print("\n--- what the model actually outputs ---")
print(f"  predict_proba shape {balanced.predict_proba(X_test).shape}  "
      f"(column 0 = P(slow), column 1 = P(fast))")
print(f" first 5 P(fast): {np.round(proba[:5], 3)}")
print(f" first 5 labels : {labels[:5]}")
print(f"\n  .predict() is just .predict_proba()[:,1] >= 0.5")
print(f"  agreement check: {np.array_equal(labels, (proba >= 0.5).astype(int))}")


--- what the model actually outputs ---
  predict_proba shape (513, 2)  (column 0 = P(slow), column 1 = P(fast))
 first 5 P(fast): [0.731 0.964 0.041 0.722 0.824]
 first 5 labels : [1 1 0 1 1]

  .predict() is just .predict_proba()[:,1] >= 0.5
  agreement check: True
